# Solar filament instance segmentation - reproducible pipeline

Two modes:
* `REPRODUCE_SUBMISSION` - downloads the trained checkpoints (SHA-256 verified), predicts the 180 test photos (10-model ensemble, 8x dihedral TTA), writes `submission.csv` and H-alpha overlay figures. Needs a GPU (~1 h).
* `TRAIN_FULL` - retrains all 10 models from scratch (5 folds x {1536, 2048}; 20+ GPU hours), tunes post-processing on out-of-fold predictions, then predicts.

Inference uses only the single test image; no metadata, neighbours or ground truth.

In [ ]:
RUN_MODE = "REPRODUCE_SUBMISSION"   # or "TRAIN_FULL"
DATA_ROOT = "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"  # folder with train/ and test/
WORK = "work"
TTA = 8

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import glob, subprocess, sys
import torch
print("GPU:", torch.cuda.is_available())
run = lambda *a: subprocess.run([sys.executable, "-m", *map(str, a)], check=True)

In [ ]:
if RUN_MODE == "REPRODUCE_SUBMISSION":
    run("solarfil.weights", "--manifest", "configs/weights.json", "--dest", "weights")
    CKPTS, POSTCFG = sorted(glob.glob("weights/*.pth")), "configs/postcfg.json"
elif RUN_MODE == "TRAIN_FULL":
    for size, epochs in ((1536, 36), (2048, 32)):
        for fold in range(5):
            run("solarfil.train", "--data_root", DATA_ROOT, "--work", WORK, "--fold", fold,
                "--size", size, "--crop", 768, "--epochs", epochs)
    run("solarfil.tune", "--data_root", DATA_ROOT, "--oof", f"{WORK}/oof_Unet_efficientnet-b4_s1536",
        "--out", f"{WORK}/postcfg.json")
    CKPTS, POSTCFG = sorted(glob.glob(f"{WORK}/Unet_*.pth")), f"{WORK}/postcfg.json"
else:
    raise ValueError(RUN_MODE)
print(len(CKPTS), "checkpoints")

In [ ]:
run("solarfil.infer", "--data_root", DATA_ROOT, "--ckpts", *CKPTS, "--postcfg", POSTCFG,
    "--work", WORK, "--tta", TTA, "--out", "submission.csv")

In [ ]:
from solarfil.visualize import render_submission
from IPython.display import Image, display
figs = render_submission("submission.csv", f"{DATA_ROOT}/test/test_images", "figures", n=6)
for f in figs[:3]:
    display(Image(f, width=700))